# Backbone screen: results analysis

In [7]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [8]:
import json

import pandas as pd
from IPython.display import display

from src.models.classical_baseline import HORIZONS

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results'

MODEL_FILES = {
    'Persistence':      RESULTS_DIR / 'persistence' / 'persistence_val_participant_stability_summary.json',
    'LinearRegression': RESULTS_DIR / 'linear_regression' / 'lr_val_participant_stability_summary.json',
    'XGBoost':          RESULTS_DIR / 'xgboost' / 'xgboost_val_stability_summary.json',
    'LSTM':             RESULTS_DIR / 'lstm' / 'lstm_val_stability_summary.json',
    'GRU':              RESULTS_DIR / 'gru' / 'gru_val_stability_summary.json',
    'N-BEATS':          RESULTS_DIR / 'nbeats' / 'nbeats_val_stability_summary.json',
    'N-HiTS':           RESULTS_DIR / 'nhits' / 'nhits_val_stability_summary.json',
}

METRICS = {
    'rmse': 'RMSE (mg/dL)', 'mae': 'MAE (mg/dL)', 'grmse': 'gRMSE',
    'zone_a': 'Zone A (%)', 'zone_ab': 'Zone A+B (%)',
}

In [9]:
per_horizon_summary = {}
for model_name, path in MODEL_FILES.items():
    if not path.exists():
        print(f'MISSING: {model_name} -- expected {path}')
        continue
    with open(path) as f:
        d = json.load(f)
    per_horizon_summary[model_name] = d['per_horizon_summary']
    seeds = d.get('seeds')
    print(f"{model_name:<16} seeds={seeds if seeds is not None else '(deterministic)'}")

Persistence      seeds=(deterministic)
LinearRegression seeds=(deterministic)
XGBoost          seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
LSTM             seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
GRU              seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
N-BEATS          seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
N-HiTS           seeds=[7, 14, 21, 28, 35, 42, 49, 56, 63, 70]


In [10]:
def format_cell(entry: dict) -> str:
    return (f"{entry['mean']:.2f} \u00b1 {entry['sd1_across_seeds']:.2f} "
            f"({entry['sd2_across_participants']:.2f})")

tables = {}
for metric_key, metric_label in METRICS.items():
    rows = {}
    for model_name in MODEL_FILES:
        if model_name not in per_horizon_summary:
            continue
        rows[model_name] = {
            f'{h}min': format_cell(per_horizon_summary[model_name][str(h)][metric_key])
            for h in HORIZONS
        }
    table = pd.DataFrame(rows).T
    table = table[[f'{h}min' for h in HORIZONS]]  # enforce horizon column order
    tables[metric_key] = table

    print(f'=== {metric_label}: mean \u00b1 sd1 (sd2) ===')
    display(table)
    print()

=== RMSE (mg/dL): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
Persistence,12.31 ± 0.00 (2.45),19.40 ± 0.00 (4.20),27.95 ± 0.00 (8.31),33.72 ± 0.00 (11.48),37.80 ± 0.00 (13.33)
LinearRegression,11.43 ± 0.00 (2.45),18.39 ± 0.00 (4.05),25.86 ± 0.00 (7.78),30.25 ± 0.00 (10.82),32.91 ± 0.00 (12.62)
XGBoost,11.36 ± 0.03 (2.60),18.01 ± 0.01 (4.14),25.58 ± 0.02 (7.81),29.82 ± 0.03 (10.49),32.35 ± 0.03 (12.08)
LSTM,11.04 ± 0.10 (2.47),17.55 ± 0.09 (3.85),24.98 ± 0.09 (7.39),29.29 ± 0.12 (10.30),31.86 ± 0.17 (12.00)
GRU,10.96 ± 0.09 (2.38),17.48 ± 0.10 (3.80),24.89 ± 0.08 (7.35),29.16 ± 0.10 (10.25),31.76 ± 0.11 (12.02)
N-BEATS,11.15 ± 0.06 (2.36),17.60 ± 0.05 (3.83),25.04 ± 0.04 (7.42),29.32 ± 0.04 (10.34),31.90 ± 0.04 (12.17)
N-HiTS,10.88 ± 0.06 (2.26),17.46 ± 0.04 (3.74),24.96 ± 0.03 (7.47),29.27 ± 0.02 (10.38),31.84 ± 0.04 (12.15)



=== MAE (mg/dL): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
Persistence,8.15 ± 0.00 (1.65),12.74 ± 0.00 (2.77),18.27 ± 0.00 (5.79),22.16 ± 0.00 (8.35),25.30 ± 0.00 (10.14)
LinearRegression,7.74 ± 0.00 (1.63),12.42 ± 0.00 (2.61),17.42 ± 0.00 (4.98),20.53 ± 0.00 (7.19),22.62 ± 0.00 (8.67)
XGBoost,7.58 ± 0.02 (1.73),11.90 ± 0.01 (2.63),17.01 ± 0.01 (5.03),20.19 ± 0.02 (6.90),22.29 ± 0.02 (8.05)
LSTM,7.45 ± 0.11 (1.66),11.73 ± 0.14 (2.47),16.68 ± 0.21 (4.70),19.81 ± 0.23 (6.77),21.93 ± 0.29 (7.99)
GRU,7.40 ± 0.09 (1.60),11.65 ± 0.09 (2.43),16.62 ± 0.13 (4.67),19.73 ± 0.16 (6.69),21.84 ± 0.15 (8.03)
N-BEATS,7.58 ± 0.07 (1.60),11.73 ± 0.12 (2.45),16.65 ± 0.22 (4.72),19.75 ± 0.22 (6.89),21.88 ± 0.19 (8.34)
N-HiTS,7.31 ± 0.09 (1.48),11.60 ± 0.09 (2.37),16.56 ± 0.13 (4.79),19.66 ± 0.15 (6.90),21.80 ± 0.18 (8.33)



=== gRMSE: mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
Persistence,13.31 ± 0.00 (3.31),21.40 ± 0.00 (5.93),31.58 ± 0.00 (11.25),38.34 ± 0.00 (15.01),42.99 ± 0.00 (17.38)
LinearRegression,12.40 ± 0.00 (3.32),20.73 ± 0.00 (6.12),30.66 ± 0.00 (11.94),36.63 ± 0.00 (16.17),40.26 ± 0.00 (18.85)
XGBoost,12.50 ± 0.03 (3.66),20.50 ± 0.02 (6.33),30.48 ± 0.03 (11.89),36.14 ± 0.04 (15.71),39.50 ± 0.04 (18.19)
LSTM,12.04 ± 0.20 (3.39),19.81 ± 0.13 (5.78),29.59 ± 0.11 (11.19),35.41 ± 0.14 (15.33),38.86 ± 0.19 (17.96)
GRU,11.92 ± 0.18 (3.26),19.69 ± 0.17 (5.68),29.43 ± 0.16 (11.02),35.19 ± 0.17 (15.14),38.69 ± 0.19 (17.87)
N-BEATS,12.19 ± 0.14 (3.28),19.91 ± 0.15 (5.78),29.64 ± 0.18 (11.16),35.35 ± 0.19 (15.26),38.81 ± 0.16 (18.00)
N-HiTS,11.81 ± 0.11 (3.10),19.69 ± 0.14 (5.65),29.55 ± 0.12 (11.24),35.33 ± 0.10 (15.33),38.75 ± 0.10 (18.01)



=== Zone A (%): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
Persistence,95.35 ± 0.00 (3.80),87.74 ± 0.00 (4.96),78.68 ± 0.00 (5.59),72.64 ± 0.00 (8.27),67.65 ± 0.00 (10.28)
LinearRegression,96.15 ± 0.00 (3.72),88.49 ± 0.00 (6.49),80.31 ± 0.00 (5.67),74.88 ± 0.00 (7.71),71.60 ± 0.00 (9.09)
XGBoost,96.29 ± 0.05 (3.82),89.42 ± 0.05 (5.72),80.85 ± 0.07 (5.71),75.27 ± 0.09 (7.41),71.68 ± 0.09 (8.33)
LSTM,96.78 ± 0.13 (3.08),89.77 ± 0.21 (5.65),81.59 ± 0.32 (5.36),76.39 ± 0.48 (7.26),72.70 ± 0.46 (8.08)
GRU,96.71 ± 0.16 (3.08),89.91 ± 0.22 (5.56),81.69 ± 0.23 (5.35),76.59 ± 0.35 (7.10),72.88 ± 0.35 (8.15)
N-BEATS,96.59 ± 0.14 (3.33),89.81 ± 0.19 (5.43),81.61 ± 0.40 (5.11),76.55 ± 0.45 (7.27),72.88 ± 0.47 (8.37)
N-HiTS,96.75 ± 0.11 (3.17),89.97 ± 0.12 (5.34),81.80 ± 0.25 (5.23),76.72 ± 0.32 (7.58),72.97 ± 0.45 (8.32)



=== Zone A+B (%): mean ± sd1 (sd2) ===


,15min,30min,60min,90min,120min
Persistence,99.79 ± 0.00 (0.36),99.43 ± 0.00 (0.67),98.60 ± 0.00 (1.56),97.74 ± 0.00 (2.57),97.08 ± 0.00 (3.13)
LinearRegression,99.78 ± 0.00 (0.40),99.38 ± 0.00 (0.80),98.72 ± 0.00 (1.44),98.17 ± 0.00 (2.05),97.81 ± 0.00 (2.57)
XGBoost,99.55 ± 0.01 (0.92),99.23 ± 0.01 (1.12),98.68 ± 0.01 (1.48),98.17 ± 0.02 (2.02),97.88 ± 0.02 (2.44)
LSTM,99.69 ± 0.03 (0.60),99.34 ± 0.02 (0.96),98.68 ± 0.02 (1.44),98.17 ± 0.03 (2.03),97.82 ± 0.05 (2.59)
GRU,99.66 ± 0.04 (0.68),99.33 ± 0.03 (0.97),98.69 ± 0.02 (1.43),98.18 ± 0.03 (2.00),97.89 ± 0.07 (2.47)
N-BEATS,99.64 ± 0.03 (0.71),99.32 ± 0.03 (0.98),98.68 ± 0.02 (1.45),98.20 ± 0.03 (1.97),97.88 ± 0.04 (2.49)
N-HiTS,99.71 ± 0.02 (0.59),99.34 ± 0.02 (0.95),98.67 ± 0.01 (1.46),98.19 ± 0.02 (1.98),97.92 ± 0.03 (2.40)


In [11]:
rmse_means = pd.DataFrame({
    model_name: {f'{h}min': per_horizon_summary[model_name][str(h)]['rmse']['mean']
                 for h in HORIZONS}
    for model_name in per_horizon_summary
}).T[[f'{h}min' for h in HORIZONS]].round(2)

print('=== RMSE (mg/dL), mean only -- for a quick ranking scan ===')
display(rmse_means)

=== RMSE (mg/dL), mean only -- for a quick ranking scan ===


,15min,30min,60min,90min,120min
Persistence,12.31,19.40,27.95,33.72,37.80
LinearRegression,11.43,18.39,25.86,30.25,32.91
XGBoost,11.36,18.01,25.58,29.82,32.35
LSTM,11.04,17.55,24.98,29.29,31.86
GRU,10.96,17.48,24.89,29.16,31.76
N-BEATS,11.15,17.60,25.04,29.32,31.90
N-HiTS,10.88,17.46,24.96,29.27,31.84


In [12]:
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / '2_best_cgm_encoder' / 'results'/ 'cgm_encoder'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for metric_key, table in tables.items():
    out_path = OUTPUT_DIR / f'backbone_screen_val_{metric_key}_table.csv'
    table.to_csv(out_path)
    print(f'saved {out_path}')

rmse_means_path = OUTPUT_DIR / 'backbone_screen_val_rmse_means.csv'
rmse_means.to_csv(rmse_means_path)
print(f'saved {rmse_means_path}')

saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_rmse_table.csv
saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_mae_table.csv
saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_grmse_table.csv
saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_zone_a_table.csv
saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_zone_ab_table.csv
saved /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoder/results/cgm_encoder/backbone_screen_val_rmse_means.csv
